# Lesson 4B: Temporal Difference Learning - Practical

## Introduction

In Lesson 4A, we learned TD theory: bootstrapping, TD error, Sarsa, Q-learning.

Now we implement and compare them on **CliffWalking**—an environment that highlights the on-policy vs off-policy difference.

### CliffWalking Problem

- 4×12 grid world
- Start: bottom-left
- Goal: bottom-right
- Cliff: dangerous path (−100 reward, reset)
- Safe path: longer but safe (+−1 per step)

**Key insight**: On-policy (Sarsa) avoids cliff; off-policy (Q-learning) cuts through it.

## Quick Setup

In [ ]:
import subprocess, sys
for pkg in ['numpy', 'matplotlib', 'gymnasium']:
    try:
        __import__(pkg)
    except:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from collections import defaultdict

np.random.seed(42)
print('✅ Ready for TD Learning')

## Sarsa vs Q-Learning Comparison

In [ ]:
def sarsa(env, episodes=500, alpha=0.1, gamma=0.9, epsilon=0.1):
    """On-policy TD learning (Sarsa)."""
    Q = defaultdict(lambda: defaultdict(float))
    rewards = []
    
    for _ in range(episodes):
        obs, _ = env.reset()
        action = np.random.randint(4) if np.random.random() < epsilon else np.argmax([Q[obs][a] for a in range(4)])
        episode_reward = 0
        done = False
        
        while not done:
            obs_next, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            episode_reward += reward
            
            action_next = np.random.randint(4) if np.random.random() < epsilon else np.argmax([Q[obs_next][a] for a in range(4)])
            td_error = reward + gamma * Q[obs_next][action_next] - Q[obs][action]
            Q[obs][action] += alpha * td_error
            
            obs, action = obs_next, action_next
        
        rewards.append(episode_reward)
    
    return np.array(rewards), Q


def qlearning(env, episodes=500, alpha=0.1, gamma=0.9, epsilon=0.1):
    """Off-policy TD learning (Q-learning)."""
    Q = defaultdict(lambda: defaultdict(float))
    rewards = []
    
    for _ in range(episodes):
        obs, _ = env.reset()
        episode_reward = 0
        done = False
        
        while not done:
            action = np.random.randint(4) if np.random.random() < epsilon else np.argmax([Q[obs][a] for a in range(4)])
            obs_next, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            episode_reward += reward
            
            max_next = max([Q[obs_next][a] for a in range(4)])
            td_error = reward + gamma * max_next - Q[obs][action]
            Q[obs][action] += alpha * td_error
            
            obs = obs_next
        
        rewards.append(episode_reward)
    
    return np.array(rewards), Q


# Run on CliffWalking
env = gym.make('CliffWalking-v0')
print('Running Sarsa and Q-Learning on CliffWalking...')

rewards_sarsa, Q_sarsa = sarsa(env, episodes=500)
rewards_ql, Q_ql = qlearning(env, episodes=500)

print(f'✅ Sarsa mean reward: {np.mean(rewards_sarsa[-50:]):.1f}')
print(f'✅ Q-Learning mean reward: {np.mean(rewards_ql[-50:]):.1f}')

## Visualization

In [ ]:
# Plot learning curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Learning curves
window = 10
ax1.plot(np.convolve(rewards_sarsa, np.ones(window)/window, mode='valid'), label='Sarsa (On-Policy)', linewidth=2)
ax1.plot(np.convolve(rewards_ql, np.ones(window)/window, mode='valid'), label='Q-Learning (Off-Policy)', linewidth=2)
ax1.set_xlabel('Episode', fontsize=11)
ax1.set_ylabel('Episode Reward', fontsize=11)
ax1.set_title('Learning Curves (Smoothed)', fontsize=12, fontweight='bold')
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# Final performance
ax2.bar(['Sarsa', 'Q-Learning'], 
        [np.mean(rewards_sarsa[-50:]), np.mean(rewards_ql[-50:])],
        color=['steelblue', 'coral'])
ax2.set_ylabel('Mean Reward (Last 50 Episodes)', fontsize=11)
ax2.set_title('Final Performance Comparison', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('\nKey Insight:')
print('- Sarsa: Conservative (avoids cliff) → safer path')
print('- Q-Learning: Aggressive (learns cliff optimal) → faster route')
print('- Both converge, but different strategies!')

## Hyperparameter Sensitivity

In [ ]:
# Quick alpha sensitivity
alphas = [0.01, 0.05, 0.1, 0.2]
fig, ax = plt.subplots(figsize=(10, 5))

for alpha in alphas:
    _, Q = qlearning(env, episodes=300, alpha=alpha)
    # Estimate value of start state
    start_val = max([Q[0][a] for a in range(4)])
    ax.text(alpha, start_val, f'α={alpha}', fontsize=10)

ax.set_xlabel('Learning Rate (α)', fontsize=11)
ax.set_ylabel('Estimated Value of Start State', fontsize=11)
ax.set_title('Impact of Learning Rate on Q-Learning', fontsize=12, fontweight='bold')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print('Observation: Higher α → faster convergence but more noise')

## Summary

### Key Findings

1. **Sarsa** (on-policy): Learns safe policy, avoids risky cliff
2. **Q-Learning** (off-policy): Learns optimal policy via risky cliff
3. **Convergence**: Both reach stable performance but with different strategies
4. **Learning rate**: Higher α converges faster but less stable

### Why This Matters

- Real systems often can't afford to crash (Sarsa safety)
- But we want optimal performance (Q-learning optimality)
- TD methods balance both: bootstrap keeps variance low, sampling keeps model-free

### Next: Lesson 5

**N-step methods & Function Approximation**—extend TD to multi-step returns and large state spaces.